# Querying Files

`query_files` is the primary way to search and retrieve files stored in DataLab. You can filter by filename using glob patterns, by tags, or combine both. Every file object in the result includes an `id` field, which is the unique identifier you will use in later tutorials to download, delete, or feed files into a pipeline.

## Setup

In [ ]:
import gfhub

# Reads host and API key from ~/.gdsfactory/gdsfactoryplus.toml or environment variables.
# You can also pass them explicitly: gfhub.Client(host="http://...", api_key="...")
client = gfhub.Client()

## Get all files

Calling `query_files()` with no arguments returns every file in your organization.

In [ ]:
files = client.query_files()

print(f"Total files: {len(files)}")
for f in files[:5]:
    print(f"  - {f['original_name']}")

## Inspect a file object

Each entry in the result is a dictionary with the following key fields:

| Field | Description |
|---|---|
| `id` | Unique identifier. Use this to download, delete, or trigger pipelines |
| `name` / `original_name` | Filename as uploaded |
| `mime_type` | Detected MIME type (e.g. `text/csv`, `application/octet-stream`) |
| `status` | File availability (e.g. `Available`) |
| `file_size` | File size in bytes |
| `created_at` | Upload timestamp (ISO 8601) |
| `tags` | Dict of tags keyed by tag name. Each tag has `id`, `name`, `color`, and optionally `parameter_value` for parameter tags |
| `pipelines` | List of pipelines this file is connected to. Each entry has an `id` and `name` (covered in the pipelines tutorial) |

In [ ]:
import json

if files:
    print(json.dumps(files[0], indent=2, default=str))

## Filter by name

The `name` parameter supports glob patterns (case-insensitive). Use `*` to match any sequence of characters.

In [ ]:
# Exact match (case-insensitive)
files = client.query_files(name="lattice.gds")
print(f"Exact match: {len(files)} file(s)")
for f in files:
    print(f"  - {f['original_name']}")

In [ ]:
# Glob patterns
for pattern in ["*.gds", "waveguide*.csv"]:
    matches = client.query_files(name=pattern)
    print(f"  '{pattern}' → {len(matches)} file(s)")

## Filter by tags

Tags are labels attached to files at upload time. Simple tags are plain labels like `"raw"` or `"reviewed"`. Parameter tags carry a value in the format `"key:value"`, for example `"wafer_id:wafer1"`. To query files that have a parameter tag regardless of its value, use just the key: `"wafer_id"`. To filter for a specific value, use the full form: `"wafer_id:wafer1"`.

Extension tags like `.gds`, `.csv`, and `.parquet` are applied automatically based on file type.

When you pass multiple tags, a file must have all of them to be returned.

In [ ]:
# Filter by file extension tag (auto-applied at upload)
for ext in [".gds", ".csv", ".parquet"]:
    matches = client.query_files(tags=[ext])
    print(f"  '{ext}' → {len(matches)} file(s)")

In [ ]:
# Filter by a named tag (applied manually at upload)
files = client.query_files(tags=["components"])
print(f"Files tagged 'raw': {len(files)}")
for f in files[:5]:
    print(f"  - {f['original_name']}")

In [ ]:
# Parameter tag — any value (query by tag name alone)
files = client.query_files(tags=["wafer_id"])
print(f"Files with any wafer_id: {len(files)}")
for f in files:
    tag = f["tags"].get("wafer_id", {})
    wafer_value = tag.get("parameter_value", "?")
    print(f"  - {f['original_name']}  (wafer_id={wafer_value})")

# Parameter tag — exact value
files = client.query_files(tags=["wafer_id:1"])
print(f"\nFiles for wafer1: {len(files)}")
for f in files:
    print(f"  - {f['original_name']}")

## Combine name and tag filters

Both filters can be used together. Only files that match the name pattern and have all the specified tags will be returned.

In [ ]:
# Parquet measurement files for a specific wafer
files = client.query_files(name="*.csv", tags=["wafer_id:1"])

print(f"Found {len(files)} file(s):")
for f in files:
    tag_names = list(f["tags"].keys())
    print(f"  - {f['original_name']}  (id={f['id']}, tags={tag_names})")